# 🎾 OMNIS-COURT LLM + Jina Server
## Colab Primary Instance

**Steps:**
1. Runtime → Change runtime type → T4 GPU
2. Run All
3. Wait ~3-5 min for model download
4. Copy both URLs when printed
5. Paste into config/platforms.json
6. Close tab safely (anti-idle active)

In [ ]:
# ==========================================
# CELL 1: INSTALL DEPENDENCIES (SAFE FIX)
# ==========================================

# Force CUDA platform BEFORE anything else
import os
os.environ['VLLM_TARGET_DEVICE'] = 'cuda'
os.environ['VLLM_PLATFORM'] = 'cuda'

# Clean ALL potentially conflicting packages
!pip uninstall -y vllm flashinfer-python humming-kernels xformers \
  compressed-tensors outlines outlines_core xgrammar llguidance 2>/dev/null

# Install vLLM 0.8.4 (stable on Colab T4 + CUDA 12.8)
!pip install "vllm==0.8.4" --no-cache-dir 2>&1 | tail -10

# Install other packages
!pip install -q trafilatura fastapi uvicorn nest-asyncio

# Install cloudflared binary (NOT a pip package)
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify ALL installations
import subprocess
cf = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'\n✅ cloudflared: {cf.stdout.strip()}')

all_ok = True
for pkg in ['vllm', 'trafilatura', 'fastapi', 'uvicorn']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'✅ {pkg}')
    except Exception as e:
        print(f'❌ {pkg}: {e}')
        all_ok = False

if all_ok:
    print('\n✅ ALL dependencies ready!')
    print('⚠️ IMPORTANT: Runtime → Restart runtime → Then Run All again')
else:
    print('\n❌ SOME packages failed. DO NOT proceed. Send error above.')

In [ ]:
# ==========================================
# CELL 2: ANTI-IDLE + ENV VARS
# ==========================================

# MUST set env vars after every restart
import os
os.environ['VLLM_TARGET_DEVICE'] = 'cuda'
os.environ['VLLM_PLATFORM'] = 'cuda'

from IPython.display import display, Javascript

display(Javascript('''
    setInterval(function(){
        var b=document.querySelector("colab-run-button");
        if(b)b.click();
    },300000);
'''))
print('✅ Anti-idle active! Safe to close tab after all cells run.')
print('✅ CUDA platform forced.')

In [ ]:
# ==========================================
# CELL 3: START QWEN2.5 VLLM SERVER
# ==========================================
import subprocess, time, requests

# Ensure env vars are set
import os
os.environ['VLLM_TARGET_DEVICE'] = 'cuda'
os.environ['VLLM_PLATFORM'] = 'cuda'

proc = subprocess.Popen([
    'python','-m','vllm.entrypoints.openai.api_server',
    '--model','Qwen/Qwen2.5-32B-Instruct',
    '--served-model-name','qwen-llm',
    '--host','0.0.0.0','--port','8000',
    '--max-model-len','8192',
    '--gpu-memory-utilization','0.9',
    '--trust-remote-code','--enforce-eager'
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print('🚀 Starting vLLM with Qwen2.5-32B... (~3-5 min)')
for i in range(30):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ vLLM READY on port 8000 ({(i+1)*10}s)')
            break
    except:
        pass
    time.sleep(10)
    if i % 3 == 0:
        print(f'⏳ Waiting... {(i+1)*10}s')
else:
    print('❌ Failed after 300s. Last logs:')
    print(proc.stderr.read().decode()[-1000:])

In [ ]:
# ==========================================
# CELL 4: START JINA READER SERVER
# ==========================================
import threading, time, requests as req
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn, nest_asyncio
nest_asyncio.apply()

app = FastAPI()

@app.get('/health')
async def health():
    return {'status':'ok'}

@app.get('/extract')
async def extract(url: str = Query(...)):
    try:
        dl = trafilatura.fetch_url(url)
        if not dl:
            return JSONResponse(400, content={'error':'fetch failed','url':url})
        txt = trafilatura.extract(dl, include_comments=False, include_tables=True, no_fallback=False)
        if not txt or len(txt.strip()) < 50:
            return JSONResponse(400, content={'error':'content too short','url':url})
        return {'url':url,'content':txt,'word_count':len(txt.split()),'status':'success'}
    except Exception as e:
        return JSONResponse(500, content={'error':str(e),'url':url})

def run():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(3)
try:
    r = req.get('http://localhost:8001/health', timeout=5)
    print('✅ Jina Reader READY on port 8001' if r.status_code==200 else '❌ Jina error')
except Exception as e:
    print(f'❌ Jina failed: {e}')

In [ ]:
# ==========================================
# CELL 5: CLOUDFLARE TUNNELS
# ==========================================
import subprocess, re

def tunnel(port):
    p = subprocess.Popen(
        ['cloudflared','tunnel','--url',f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return p, m.group(0)
    return p, None

print('🌐 Tunnel LLM (8000)...')
p1, u1 = tunnel(8000)
print('🌐 Tunnel Jina (8001)...')
p2, u2 = tunnel(8001)

if u1 and u2:
    print('\n' + '='*60)
    print('🎉 OMNIS-COURT COLAB READY!')
    print('='*60)
    print(f'🧠 LLM:  {u1}')
    print(f'📖 JINA: {u2}')
    print('='*60)
    print('📋 COPY BOTH URLs → config/platforms.json')
    print('🔒 Anti-idle ON → safe to close tab')
else:
    print(f'❌ Tunnel failed: LLM={u1}, Jina={u2}')

In [ ]:
# ==========================================
# CELL 6: TEST BOTH SERVICES
# ==========================================
import requests

llm_ok = False
jina_ok = False

print('🧪 Testing LLM (local)...')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={'model':'qwen-llm','messages':[{'role':'user','content':'Say OK'}],'max_tokens':5},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ LLM Local: {r.json()['choices'][0]['message']['content']}")
        llm_ok = True
    else:
        print(f'❌ LLM Local: {r.status_code}')
except Exception as e:
    print(f'❌ LLM Local: {e}')

print('🧪 Testing Jina (local)...')
try:
    r = requests.get(
        'http://localhost:8001/extract',
        params={'url':'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ Jina Local: {r.json()['word_count']} words")
        jina_ok = True
    else:
        print(f'❌ Jina Local: {r.status_code}')
except Exception as e:
    print(f'❌ Jina Local: {e}')

print()
if llm_ok and jina_ok:
    print('✅ ALL LOCAL TESTS PASSED!')
    print('📋 Now test tunnel URLs from browser:')
    print(f'   LLM:  {u1}/v1/models')
    print(f'   Jina: {u2}/health')
else:
    print('❌ Some local tests failed. Check errors above.')